# 03 — Final analysis

Run all cells with Internet access on Kaggle, Colab, or local Jupyter. The notebook downloads the public team repository when needed and installs its single `requirements.txt`. A CPU is sufficient: no pretrained weights, dataset download, or GPU is needed.

Source tables: `results/02_second_models` (all four models). Quality/runtime tables and comparison charts are rebuilt from these inputs. The three previously exported failure cases are reused from `results/03_final_analysis/failure_cases.csv` and its PNG figures; their query/gallery IDs, breed labels, cosine scores, and case-level P@5/Hit@5 are validated against the shared manifest. Complete historical rankings are not part of the repository and are not recomputed here. Runtime values come exclusively from the Tesla T4 run of 7 November 2026.

The analysis uses three scripts and the shared `src/` modules. It writes three CSV files, one JSON, and five PNG files in `results/03_final_analysis/`. A ZIP containing only those nine files is saved **next to the repository directory** as `final_analysis_results.zip`.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPOSITORY_URL = "https://github.com/kite121/Visual_retrieval_system.git"
BRANCH = "main"


def run(command, cwd=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)


start_dir = Path.cwd().resolve()
local_repositories = [
    parent for parent in (start_dir, *start_dir.parents)
    if (parent / ".git").exists()
    and (parent / "scripts" / "run_evaluation.py").is_file()
]
if local_repositories:
    ROOT = local_repositories[0]
    print("Using the existing repository checkout:", ROOT)
else:
    ROOT = start_dir / "Visual_retrieval_system"
    if not ROOT.exists():
        run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPOSITORY_URL, ROOT])
    else:
        if not (ROOT / ".git").exists():
            raise RuntimeError(f"{ROOT} exists but is not a Git checkout")
        current_branch = subprocess.check_output(
            ["git", "branch", "--show-current"], cwd=ROOT, text=True,
        ).strip()
        if current_branch != BRANCH:
            raise RuntimeError(f"Switch {ROOT} to {BRANCH} before running this notebook")
        run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=ROOT)

os.chdir(ROOT)
SOURCE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True,
).strip()
print("Repository:", ROOT)
print("Source commit:", SOURCE_COMMIT)

required = [
    "requirements.txt", "configs/split_manifest.csv",
    "scripts/build_final_results.py", "scripts/generate_final_figures.py",
    "scripts/export_failure_cases.py",
    "results/03_final_analysis/failure_cases.csv", "results/03_final_analysis/run_metadata.json",
    *[f"results/03_final_analysis/figures/failure_case_{i:02d}.png" for i in range(1, 4)],
    *[f"results/02_second_models/{name}" for name in (
        "metrics_validation.csv", "metrics_test.csv", "runtime_summary.csv", "run_metadata.json",
    )],
]
missing = [name for name in required if not (ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(
        "The checkout is missing final-analysis inputs: " + ", ".join(missing)
        + ". Update this checkout to the current main branch."
    )

## Environment setup

Install dependencies from the repository root. Existing compatible installations are retained. All later cells import and execute the repository scripts; this notebook contains no duplicated implementation of the analysis.

In [ ]:
run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-q", "-r", ROOT / "requirements.txt",
])

from IPython.display import FileLink, Image, display
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT))
from build_final_results import build
from generate_final_figures import generate
from export_failure_cases import export

SOURCE = ROOT / "results/02_second_models"
OUTPUT = ROOT / "results/03_final_analysis"
print("Analysis source:", SOURCE)
print("Output directory:", OUTPUT)

## 1. Таблицы и происхождение данных

Сохраняются исходные имена столбцов нового формата. Проверяются SHA-256 манифеста, размеры выборок, четыре pretrained-модели, диапазоны метрик и статистики трёх повторов. Данные `01_first_models` не добавляются повторно: они уже включены в `02_second_models`.

In [ ]:
quality, runtime = build(SOURCE, OUTPUT)
display(quality)
display(runtime)

## 2. Сравнительные графики

Test Precision@5 и время. Gallery build показано в секундах, компоненты запроса — в миллисекундах. Погрешности показывают min/max повторов, а не доверительные интервалы. Медианы компонентов не складываются в якобы измеренную общую задержку.

In [ ]:
from IPython.display import Image, display
generate(OUTPUT)
display(Image(filename=str(OUTPUT / "figures/precision_at_5_comparison.png")))
display(Image(filename=str(OUTPUT / "figures/runtime_comparison.png")))

## 3. Три сохранённых диагностических ошибки

Используются готовые CSV и PNG трёх примеров из отдельного прогона от 31 октября 2026. Проверяются test-query ID, принадлежность соседей галерее, породы, порядок cosine scores и соответствие P@5/Hit@5 сохранённому Top-5. Гипотезы о причинах отделены от наблюдений.

Полные исторические rankings и исходные изображения больше не включены в репозиторий: эта ячейка проверяет сохранённые записи и показывает готовые figures. Она не повторяет полный retrieval и не генерирует изображения заново. Метаданные разделяют текущие проверки трёх примеров и результаты ранее выполненной проверки полного test.

In [ ]:
failures = export(OUTPUT)
display(failures.drop(columns="hits_json"))
for filename in failures.figure:
    display(Image(filename=str(OUTPUT / filename)))

## 4. Проверка комплекта

В запросе перечислены девять файлов: три CSV, один JSON и пять PNG. Notebook можно запускать повторно: вычисляемые файлы перезаписываются, исходные результаты остаются источником анализа.

In [ ]:
import json
expected = ["final_quality_metrics.csv", "runtime_summary.csv", "failure_cases.csv", "run_metadata.json",
            "figures/precision_at_5_comparison.png", "figures/runtime_comparison.png",
            *[f"figures/failure_case_{i:02d}.png" for i in range(1, 4)]]
assert all((OUTPUT / name).is_file() for name in expected)
assert len(quality) == 8 and len(runtime) == 4 and len(failures) == 3
metadata = json.loads((OUTPUT / "run_metadata.json").read_text(encoding="utf-8"))
assert metadata["failure_cases"]["status"] == "complete"
for name in expected:
    print(f"{name}: {(OUTPUT / name).stat().st_size:,} bytes")

## Download the results

The ZIP is written beside the repository, matching notebooks 01 and 02. It contains only the nine final outputs: the four CSV/JSON files are placed directly in its root, with the five PNG files under `figures/`. The archive has no `results/` wrapper or `configs/` folder. The repository keeps its organized working outputs in `results/03_final_analysis/`.

In [ ]:
import zipfile

metadata["analysis_source_commit"] = SOURCE_COMMIT
(OUTPUT / "run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
archive_path = ROOT.parent / "final_analysis_results.zip"
files = [OUTPUT / name for name in expected]
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(OUTPUT).as_posix())
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(expected)
    assert len(archive.namelist()) == 9
print("Created:", archive_path)
display(FileLink(os.path.relpath(archive_path, Path.cwd())))